# nanoGPT language modeling

Sven vs the modern LM optimizers (AdamW/Muon/SOAP) on char-level tiny-shakespeare. Validation loss -> perplexity vs epoch and wall-time.

> Loads the fresh Gram-backend results. Robust to partial data (plots whatever has finished).

In [ ]:
import sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '.')
from style import load_results, average_over_seeds, set_style
from analysis_helpers import add_derived, best_per_method, valid, loss_curve, steps_to_target, method_order
from pathlib import Path
set_style()
PLOT_DIR = Path('plots/nanogpt'); PLOT_DIR.mkdir(parents=True, exist_ok=True)
def sven_color(m): return 'k' if m=='Sven' else None
def sven_lw(m):    return 2.6 if m=='Sven' else 1.6

In [ ]:
df = add_derived(load_results('exp_nanogpt_speedrun'))
best = best_per_method(df, by='final_val_loss')

### Best val-loss curve per optimizer (Sven in black)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.4))
for m in method_order(best['method'].unique()):
    row=best[best.method==m].iloc[0]
    cv=loss_curve(row,'val'); ct=loss_curve(row,'train')
    if cv: axes[0].plot(range(len(cv)), cv, color=sven_color(m), lw=sven_lw(m), label=m)
    if ct: axes[1].plot(range(len(ct)), ct, color=sven_color(m), lw=sven_lw(m), label=m)
axes[0].set_title('val loss'); axes[1].set_title('train loss')
for ax in axes: ax.set_xlabel('epoch'); ax.set_ylabel('cross-entropy'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(PLOT_DIR/'nanogpt_curves.pdf',bbox_inches='tight'); plt.show()

### Summary (best config per optimizer)

In [ ]:
b=best_per_method(df, by='final_val_loss').sort_values('final_val_loss')
print(f'{"optimizer":10s}{"val loss":>10s}{"val ppl":>9s}{"wall(s)":>9s}')
for _,r in b.iterrows():
    t=r.total_time if r.total_time==r.total_time else float('nan')
    print(f'{r.method:10s}{r.final_val_loss:10.4f}{r.val_ppl:9.2f}{t:9.0f}')